# Extracción de features (forma, color, textura)

Lee `processed/manifest.csv`, y para cada imagen carga `_preprocessed.png` (sin blur) y `_mask.png`.
Guarda **un CSV por split**: `processed/features_train.csv` y `processed/features_val.csv`.

- La **primera columna es `name`** (nombre de la imagen). Después: `class`, `subtype`, `label` (nevus=0, others=1)
  y dos flags de calidad `qc_*` (no son features: no las uses para entrenar).
- Las features llevan prefijo `shape_`, `color_` o `tex_` para poder hacer ablación por familias.
- Si la máscara de una imagen está vacía o falla una familia, esas columnas quedan en NaN.

In [1]:
import traceback
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import pdist
from scipy.stats import kurtosis, skew
from skimage.feature import blob_log, graycomatrix, local_binary_pattern
from skimage.filters import sato
from sklearn.cluster import KMeans
from tqdm.auto import tqdm

try:
    import mahotas
    HAS_MAHOTAS = True
except ImportError:
    HAS_MAHOTAS = False
    print("[aviso] mahotas no está instalado: se omiten los momentos de Zernike (pip install mahotas)")

warnings.filterwarnings("ignore")

# ---------------- Configuración ----------------
ROOT = Path(r"C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG")
PROC_DIR = ROOT / "preprocessed2"
N_JOBS = 4               # núcleos (-1 = todos, -2 = todos menos uno)
LIMIT = None             # p. ej. 20 para probar con pocas imágenes; None = todas
CLASS_TO_LABEL = {"nevus": 0, "others": 1}

EXTENDED_SHAPE = True    # Hu, Zernike, Fourier, curvatura, fractal, Feret... (False = solo forma básica)
KMEANS_K = 6
RIDGE_THR = 0.02         # umbral (heurístico) de respuesta del filtro de crestas Sato

In [2]:
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "train").is_dir() and (p / "val").is_dir():
            return p
    raise FileNotFoundError("No encuentro train/ y val/. Pon ROOT a mano.")


ROOT = Path(ROOT) if ROOT else find_root()
PROC = Path(PROC_DIR) if PROC_DIR else ROOT / "processed"
manifest = pd.read_csv(PROC / "manifest.csv")
manifest = manifest[manifest["status"].isin(["ok", "skipped"])].reset_index(drop=True)
if LIMIT:
    manifest = manifest.groupby("split").head(LIMIT).reset_index(drop=True)
print("processed:", PROC)
print(manifest.groupby(["split", "class"]).size().to_string())

processed: C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed2
split  class 
train  nevus     1017
       others     983
val    nevus      509
       others     491


## Utilidades

In [3]:
ell = lambda n: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (n, n))


def read_rgb(path):
    bgr = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def read_mask(path):
    return cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_GRAYSCALE) > 127


def largest_contour(mask):
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return max(cnts, key=cv2.contourArea)


def resample_contour(cnt, n=256):
    """Remuestrea el contorno por longitud de arco a n puntos (x, y)."""
    p = cnt.reshape(-1, 2).astype(np.float64)
    p = np.vstack([p, p[:1]])
    d = np.r_[0, np.cumsum(np.hypot(*np.diff(p, axis=0).T))]
    t = np.linspace(0, d[-1], n, endpoint=False)
    return np.column_stack([np.interp(t, d, p[:, 0]), np.interp(t, d, p[:, 1])])


def stats7(v, name, out):
    v = np.asarray(v, np.float64)
    out[f"{name}_mean"] = v.mean()
    out[f"{name}_std"] = v.std()
    out[f"{name}_skew"] = skew(v) if v.std() > 1e-9 else 0.0
    out[f"{name}_kurt"] = kurtosis(v) if v.std() > 1e-9 else 0.0
    p5, p50, p95 = np.percentile(v, [5, 50, 95])
    out[f"{name}_p5"], out[f"{name}_p50"], out[f"{name}_p95"] = p5, p50, p95


def skin_ring(mask, gap=10, width=60, min_px=500):
    """Anillo de piel alrededor de la lesión (a distancia entre gap y width px)."""
    dist = cv2.distanceTransform((~mask).astype(np.uint8), cv2.DIST_L2, 3)
    ring = (dist > gap) & (dist <= width)
    return ring if ring.sum() >= min_px else None


class Guard:
    """Aísla un bloque de código: si falla, anota nombre, línea y mensaje y sigue con el resto."""
    def __init__(self, errs, name):
        self.errs, self.name = errs, name

    def __enter__(self):
        return self

    def __exit__(self, et, ev, tb):
        if et is not None:
            self.errs.append(f"{self.name}:{et.__name__}@L{tb.tb_lineno}:{str(ev)[:80]}")
            return True
        return False


def err_info(e):
    """Tipo, línea y función donde falló una excepción."""
    fr = traceback.extract_tb(e.__traceback__)[-1]
    return f"{type(e).__name__}@{fr.name}:L{fr.lineno}:{str(e)[:80]}"


def principal_axes(mask):
    m = cv2.moments(mask.astype(np.uint8), binaryImage=True)
    cx, cy = m["m10"] / m["m00"], m["m01"] / m["m00"]
    mu20, mu02, mu11 = m["mu20"] / m["m00"], m["mu02"] / m["m00"], m["mu11"] / m["m00"]
    theta = 0.5 * np.arctan2(2 * mu11, mu20 - mu02)
    tmp = np.sqrt(4 * mu11 ** 2 + (mu20 - mu02) ** 2)
    l1, l2 = (mu20 + mu02 + tmp) / 2, (mu20 + mu02 - tmp) / 2
    return cx, cy, theta, max(l1, 1e-9), max(l2, 1e-9), m


def aligned(arr, cx, cy, theta, interp):
    """Rota la imagen para que el eje mayor quede horizontal y el centroide en el centro (lienzo = mitad de resolución)."""
    h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((cx, cy), np.degrees(theta), 0.5)
    M[0, 2] += w / 2 - cx
    M[1, 2] += h / 2 - cy
    return cv2.warpAffine(arr, M, (w, h), flags=interp)

## Familia 1 · Forma

In [4]:
def shape_features(img, mask):
    out = {}
    H, W = mask.shape
    m8 = mask.astype(np.uint8)
    cnt = largest_contour(mask)
    area = float(cv2.contourArea(cnt))
    perim = float(cv2.arcLength(cnt, True))
    hull = cv2.convexHull(cnt)
    hull_area = float(cv2.contourArea(hull))
    x, y, w, h = cv2.boundingRect(cnt)
    cx, cy, theta, l1, l2, mom = principal_axes(mask)
    R = np.sqrt(area / np.pi)                         # radio equivalente (normalizador de escala)

    # tamaño y compacidad
    out["shape_area_frac"] = float(mask.mean())
    out["shape_extent"] = area / (w * h)
    out["shape_circularity"] = 4 * np.pi * area / perim ** 2
    out["shape_solidity"] = area / hull_area if hull_area > 0 else np.nan
    out["shape_eccentricity"] = float(np.sqrt(1 - l2 / l1))
    out["shape_aspect_ratio"] = float(np.sqrt(l2 / l1))
    out["shape_compactness"] = perim ** 2 / (4 * np.pi * area)
    (_, _), (rw, rh), _ = cv2.minAreaRect(cnt)
    out["shape_rectangularity"] = area / max(rw * rh, 1e-9)
    out["shape_centroid_offset"] = float(np.hypot(cx - W / 2, cy - H / 2) / (min(H, W) / 2))

    # asimetría de forma (IoU con el reflejo respecto a los ejes mayor y menor)
    a = aligned(m8, cx, cy, theta, cv2.INTER_NEAREST) > 0
    out["shape_asym_major"] = (a & a[::-1, :]).sum() / max((a | a[::-1, :]).sum(), 1)   # reflejo sobre el eje mayor
    out["shape_asym_minor"] = (a & a[:, ::-1]).sum() / max((a | a[:, ::-1]).sum(), 1)   # reflejo sobre el eje menor

    # irregularidad del borde: firma radial centroide-contorno
    pts = resample_contour(cnt, 256)
    r = np.hypot(pts[:, 0] - cx, pts[:, 1] - cy)
    rn = r / r.mean()
    out["shape_radial_cv"] = float(rn.std())
    out["shape_radial_range"] = float(rn.max() - rn.min())

    # concavidades
    try:
        idx = cv2.convexHull(cnt, returnPoints=False)
        d = cv2.convexityDefects(cnt, idx)
        depth = (d.reshape(-1, 4)[:, 3] / 256.0) / R if d is not None else np.array([0.0])   # (n,4) o (n,1,4) según versión de OpenCV
    except cv2.error:
        depth = np.array([0.0])
    out["shape_n_concavities"] = int((depth > 0.05).sum())
    out["shape_concavity_max"] = float(depth.max())
    out["shape_concavity_mean"] = float(depth[depth > 0.05].mean()) if (depth > 0.05).any() else 0.0

    # nitidez del borde: gradiente en una banda alrededor del contorno
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    g = cv2.GaussianBlur(gray, (0, 0), 1.0)
    grad = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3))
    k = 5
    inner = m8 & ~cv2.erode(m8, ell(2 * k + 1)).astype(bool)
    outer = cv2.dilate(m8, ell(2 * k + 1)).astype(bool) & ~mask
    band = inner.astype(bool) | outer
    yy, xx = np.nonzero(band)
    sector = np.clip(((np.arctan2(yy - cy, xx - cx) + np.pi) / (2 * np.pi) * 8).astype(int), 0, 7)
    gv = grad[yy, xx]
    sec_g = np.array([gv[sector == s].mean() if (sector == s).any() else np.nan for s in range(8)])
    out["shape_edge_grad_mean"] = float(gv.mean())
    out["shape_edge_grad_std"] = float(gv.std())
    out["shape_edge_grad_sector_std"] = float(np.nanstd(sec_g))
    out["shape_edge_grad_sector_min"] = float(np.nanmin(sec_g))
    out["shape_edge_grad_sector_max"] = float(np.nanmax(sec_g))
    iy, ix = np.nonzero(inner.astype(bool))
    oy, ox = np.nonzero(outer)
    isec = np.clip(((np.arctan2(iy - cy, ix - cx) + np.pi) / (2 * np.pi) * 8).astype(int), 0, 7)
    osec = np.clip(((np.arctan2(oy - cy, ox - cx) + np.pi) / (2 * np.pi) * 8).astype(int), 0, 7)
    contrast = np.array([gray[iy, ix][isec == s].mean() - gray[oy, ox][osec == s].mean()
                         if (isec == s).any() and (osec == s).any() else np.nan for s in range(8)])
    out["shape_edge_contrast_mean"] = float(np.nanmean(contrast) / 255)     # <0: borde más oscuro por dentro
    out["shape_edge_contrast_sector_std"] = float(np.nanstd(contrast) / 255)

    if not EXTENDED_SHAPE:
        return out
    errs = []
    # momentos de Hu (log con signo)
    with Guard(errs, "hu"):
        hu = cv2.HuMoments(mom).ravel()
        for i, v in enumerate(hu):
            out[f"shape_hu{i + 1}"] = float(-np.sign(v) * np.log10(abs(v) + 1e-30))

    # momentos de Zernike (sobre el recorte cuadrado de la lesión a 128x128 => invariantes a escala)
    with Guard(errs, "zernike"):
        if HAS_MAHOTAS:
            s = max(w, h)
            cy0, cx0 = y + h // 2, x + w // 2
            pad = np.pad(m8, s)
            crop = pad[cy0 + s - s // 2: cy0 + s - s // 2 + s, cx0 + s - s // 2: cx0 + s - s // 2 + s]
            z = mahotas.features.zernike_moments(cv2.resize(crop, (128, 128), interpolation=cv2.INTER_NEAREST), 64, degree=8)
            for i, v in enumerate(z):
                out[f"shape_zernike{i}"] = float(v)

    # descriptores de Fourier de la firma radial (magnitudes de los 10 primeros armónicos)
    with Guard(errs, "fourier"):
        F = np.abs(np.fft.rfft(rn)) / len(rn)
        for i in range(1, 11):
            out[f"shape_fourier{i}"] = float(F[i])

    # curvatura del contorno (normalizada por el radio equivalente)
    with Guard(errs, "curvature"):
        sx = gaussian_filter1d(pts[:, 0], 3, mode="wrap")
        sy = gaussian_filter1d(pts[:, 1], 3, mode="wrap")
        dx, dy = np.gradient(sx), np.gradient(sy)
        ddx, ddy = np.gradient(dx), np.gradient(dy)
        kappa = (dx * ddy - dy * ddx) / np.maximum((dx ** 2 + dy ** 2) ** 1.5, 1e-9) * R
        kappa = kappa * np.sign(np.mean(kappa))                 # misma orientación para todos
        out["shape_curv_mean_abs"] = float(np.abs(kappa).mean())
        out["shape_curv_std"] = float(kappa.std())
        out["shape_curv_max"] = float(kappa.max())
        out["shape_curv_min"] = float(kappa.min())
        out["shape_bending_energy"] = float((kappa ** 2).mean())
        out["shape_curv_sign_changes"] = int((np.diff(np.sign(kappa)) != 0).sum())

    # diámetros de Feret (normalizados por el radio equivalente)
    with Guard(errs, "feret"):
        hp = hull.reshape(-1, 2).astype(np.float64)
        fmax = pdist(hp).max() if len(hp) > 1 else 0.0
        angs = np.deg2rad(np.arange(0, 180, 2))
        proj = hp @ np.vstack([np.cos(angs), np.sin(angs)])
        fmin = (proj.max(0) - proj.min(0)).min()
        out["shape_feret_max"] = float(fmax / R)
        out["shape_feret_min"] = float(fmin / R)
        out["shape_feret_ratio"] = float(fmin / max(fmax, 1e-9))

    # dimensión fractal del borde (box-counting) y lacunaridad
    with Guard(errs, "fractal"):
        border = (m8 - cv2.erode(m8, np.ones((3, 3), np.uint8))).astype(bool)
        sizes = [2, 4, 8, 16, 32, 64]
        counts = []
        for sz in sizes:
            hh, ww = (H // sz) * sz, (W // sz) * sz
            counts.append(border[:hh, :ww].reshape(hh // sz, sz, ww // sz, sz).any(axis=(1, 3)).sum())
        out["shape_fractal_dim"] = float(np.polyfit(np.log(1 / np.array(sizes)), np.log(np.maximum(counts, 1)), 1)[0])
        box = max(4, int(R // 4))
        sub = m8[y:y + h, x:x + w].astype(np.float32)
        f = cv2.boxFilter(sub, -1, (box, box))
        out["shape_lacunarity"] = float(1 + f.var() / max(f.mean() ** 2, 1e-9))

    if errs:
        out["_errors"] = errs
    return out

## Familia 2 · Color

In [5]:
def color_features(img, mask):
    out = {}
    lab = cv2.cvtColor(img.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)      # L 0-100, a/b ~ -127..127
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    chans = {"R": img[..., 0], "G": img[..., 1], "B": img[..., 2], "S": hsv[..., 1], "V": hsv[..., 2],
             "L": lab[..., 0], "a": lab[..., 1], "b": lab[..., 2]}
    ring = skin_ring(mask)

    for k, ch in chans.items():
        ch = ch.astype(np.float64)
        stats7(ch[mask], f"color_{k}", out)
        out[f"color_rel_{k}"] = ch[mask].mean() - ch[ring].mean() if ring is not None else np.nan   # lesión - piel

    ang = hsv[..., 0][mask].astype(np.float64) * 2 * np.pi / 180                # el matiz es circular
    out["color_hue_sin"], out["color_hue_cos"] = np.sin(ang).mean(), np.cos(ang).mean()
    out["color_hue_R"] = float(np.hypot(out["color_hue_sin"], out["color_hue_cos"]))

    # número de colores (K-means en Lab dentro de la lesión)
    px = lab[mask].astype(np.float64)
    if len(px) > 4000:
        px = px[np.random.RandomState(0).choice(len(px), 4000, replace=False)]
    km = KMeans(n_clusters=min(KMEANS_K, len(px)), n_init=3, random_state=0).fit(px)
    frac = np.bincount(km.labels_, minlength=km.n_clusters) / len(px)
    sig = frac > 0.05
    out["color_n_colors"] = int(sig.sum())
    out["color_entropy"] = float(-(frac[frac > 0] * np.log2(frac[frac > 0])).sum())
    out["color_cluster_spread"] = float(pdist(km.cluster_centers_[sig]).max()) if sig.sum() > 1 else 0.0

    # fracciones de color clínico (reglas heurísticas en Lab, excluyentes y en este orden)
    L, a, b = (lab[..., i][mask] for i in range(3))
    chroma = np.hypot(a, b)
    free = np.ones(L.shape, bool)
    rules = [("white", (L >= 75) & (chroma < 12)),
             ("black", L < 20),
             ("blue_gray", (b < 4) & (L < 70)),
             ("red", a >= 22),
             ("dark_brown", L < 38),
             ("light_brown", np.ones(L.shape, bool))]
    n_present = 0
    for name, cond in rules:
        sel = cond & free
        free &= ~sel
        out[f"color_frac_{name}"] = float(sel.mean())
        n_present += sel.mean() > 0.05
    out["color_n_clinical_colors"] = int(n_present)

    # centro vs periferia (transformada de distancia)
    dt = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 3)
    mx = dt.max()
    if mx >= 5:
        core, per = mask & (dt >= 0.6 * mx), mask & (dt <= 0.2 * mx)
        for k in ["L", "a", "b", "S"]:
            ch = chans[k].astype(np.float64)
            out[f"color_core_minus_periph_{k}"] = ch[core].mean() - ch[per].mean()
    else:
        for k in ["L", "a", "b", "S"]:
            out[f"color_core_minus_periph_{k}"] = np.nan

    # asimetría de color entre mitades (respecto a los ejes mayor y menor)
    cx, cy, theta, _, _, _ = principal_axes(mask)
    am = aligned(mask.astype(np.uint8), cx, cy, theta, cv2.INTER_NEAREST) > 0
    al = aligned(lab, cx, cy, theta, cv2.INTER_LINEAR)
    c = am.shape[0] // 2
    top, bot = am.copy(), am.copy()
    top[c:] = False
    bot[:c] = False
    lef, rig = am.copy(), am.copy()
    lef[:, c:] = False
    rig[:, :c] = False
    for nm, (m1, m2) in {"major": (top, bot), "minor": (lef, rig)}.items():
        out[f"color_asym_{nm}"] = float(np.linalg.norm(al[m1].mean(0) - al[m2].mean(0))) \
            if m1.sum() > 20 and m2.sum() > 20 else np.nan
    return out

## Familia 3 · Textura

In [6]:
def glcm_props(P):
    """P: matriz (32x32) normalizada. Devuelve las 6 propiedades."""
    n = P.shape[0]
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    mu_i, mu_j = (P * i).sum(), (P * j).sum()
    sd_i, sd_j = np.sqrt((P * (i - mu_i) ** 2).sum()), np.sqrt((P * (j - mu_j) ** 2).sum())
    asm = (P ** 2).sum()
    nz = P > 0
    return {"contrast": (P * (i - j) ** 2).sum(),
            "dissimilarity": (P * np.abs(i - j)).sum(),
            "homogeneity": (P / (1 + (i - j) ** 2)).sum(),
            "energy": np.sqrt(asm),
            "correlation": (P * (i - mu_i) * (j - mu_j)).sum() / (sd_i * sd_j) if sd_i * sd_j > 1e-9 else 0.0,
            "entropy": -(P[nz] * np.log2(P[nz])).sum()}


def texture_features(img, mask):
    out = {}
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    ys, xs = np.nonzero(mask)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    gc, mc = gray[y0:y1, x0:x1], mask[y0:y1, x0:x1]

    # GLCM: nivel 0 reservado para "fuera de la máscara" y se descarta => la frontera no cuenta
    q = (1 + (gc.astype(np.int32) * 30) // 255).astype(np.uint8)
    q[~mc] = 0
    P = graycomatrix(q, [1, 2, 4], [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=32, symmetric=True, normed=False)
    P = P.astype(np.float64)
    P[0, :, :, :] = 0
    P[:, 0, :, :] = 0
    for di, d in enumerate([1, 2, 4]):
        acc = []
        for ai in range(4):
            M = P[:, :, di, ai]
            s = M.sum()
            acc.append(glcm_props(M / s) if s > 0 else {k: np.nan for k in
                       ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "entropy"]})
        for k in acc[0]:
            out[f"tex_glcm_{k}_d{d}"] = float(np.nanmean([r[k] for r in acc]))

    # LBP uniforme (P=8, R=1) dentro de la lesión
    lbp = local_binary_pattern(gray, 8, 1, "uniform")
    inner = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    hist = np.bincount(lbp[inner].astype(int), minlength=10)[:10] / max(inner.sum(), 1)
    for i, v in enumerate(hist):
        out[f"tex_lbp{i}"] = float(v)

    # red de pigmento: filtro de crestas (Sato) sobre estructuras oscuras
    resp = sato(gc.astype(np.float32) / 255.0, sigmas=(1, 2), black_ridges=True)[mc]
    out["tex_ridge_mean"] = float(resp.mean())
    out["tex_ridge_p95"] = float(np.percentile(resp, 95))
    out["tex_ridge_frac"] = float((resp > RIDGE_THR).mean())

    # puntos/glóbulos: blobs oscuros (LoG) con centro dentro de la lesión
    blobs = blob_log(1.0 - gc.astype(np.float32) / 255.0, min_sigma=2, max_sigma=6, num_sigma=5, threshold=0.08)
    inner_c = cv2.erode(mc.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    keep = [b for b in blobs if inner_c[int(b[0]), int(b[1])]]
    out["tex_n_blobs"] = len(keep)
    out["tex_blob_density"] = len(keep) / max(mask.sum(), 1) * 1000
    out["tex_blob_sigma_mean"] = float(np.mean([b[2] for b in keep])) if keep else 0.0

    # homogeneidad
    L = cv2.cvtColor(img.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)[..., 0]
    out["tex_L_std"] = float(L[mask].std())
    out["tex_gray_std"] = float(gray[mask].std())
    h = np.bincount((gray[mask] // 8).astype(int), minlength=32) / mask.sum()
    out["tex_gray_entropy"] = float(-(h[h > 0] * np.log2(h[h > 0])).sum())
    g = gray.astype(np.float32)
    var = cv2.blur(g * g, (5, 5)) - cv2.blur(g, (5, 5)) ** 2
    out["tex_local_std_mean"] = float(np.sqrt(np.maximum(var, 0))[inner].mean())
    return out

## Extracción (paralela)

In [7]:
FAMILIES = [("shape", shape_features), ("color", color_features), ("tex", texture_features)]


def extract_one(row):
    cv2.setNumThreads(1)
    base = {"name": row["name"], "class": row["class"], "subtype": row["subtype"],
            "label": CLASS_TO_LABEL[row["class"]],
            "qc_seg_ok": row.get("seg_ok", np.nan), "qc_crop_fallback": row.get("crop_fallback", np.nan)}
    feats, err = {}, []
    try:
        img, mask = read_rgb(row["preprocessed"]), read_mask(row["mask"])
        if mask.sum() < 50:                                     # máscara vacía o degenerada
            err.append("mask_empty")
        else:
            for fam, fn in FAMILIES:
                try:
                    feats.update(fn(img, mask))
                except Exception as e:                          # una familia que falla deja NaN y no rompe el lote
                    err.append(f"{fam}:{err_info(e)}")
            err += feats.pop("_errors", [])
    except Exception as e:
        err.append(f"read:{err_info(e)}")
    return {**base, **feats, "qc_errors": ";".join(err)}


gen = Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(extract_one)(r) for r in manifest.to_dict("records"))
df = pd.DataFrame([r for r in tqdm(gen, total=len(manifest))])

meta = ["name", "class", "subtype", "label", "qc_seg_ok", "qc_crop_fallback", "qc_errors"]
cols = meta[:6] + [c for p in ("shape_", "color_", "tex_") for c in df.columns if c.startswith(p)] + ["qc_errors"]
df = df[cols]
split_of = manifest.set_index("name")["split"]
df["_split"] = df["name"].map(split_of)
print(f"\n{df.shape[0]} imágenes, {sum(c.startswith(('shape_', 'color_', 'tex_')) for c in df.columns)} features")

  0%|          | 0/3000 [00:00<?, ?it/s]


3000 imágenes, 197 features


## Guardado: un CSV por split

In [8]:
for split in ["train", "val"]:
    part = df[df["_split"] == split].drop(columns="_split")
    if len(part) == 0:
        continue
    path = PROC / f"features_{split}.csv"
    part.to_csv(path, index=False)
    print(f"{path}  ->  {part.shape[0]} filas x {part.shape[1]} columnas")

feat_cols = [c for c in df.columns if c.startswith(("shape_", "color_", "tex_"))]
nan_pct = df[feat_cols].isna().mean()
for p in ["shape", "color", "tex"]:
    cols_p = [c for c in feat_cols if c.startswith(p + "_")]
    if cols_p:
        print(f"{p:6s} {len(cols_p):3d} columnas | NaN medio: {100 * nan_pct[cols_p].mean():.2f}%")
    else:
        print(f"{p:6s}   0 columnas  <-- ¡la familia falló en todas las imágenes! mira qc_errors")
bad = df[df["qc_errors"] != ""]
print(f"Imágenes con algún error: {len(bad)}")
if len(bad):
    print(bad["qc_errors"].str.split(";").explode().str.replace(r"^(\w+:\w+@\w+:L\d+).*", r"\1", regex=True)
          .value_counts().head(10).to_string())          # errores más frecuentes (tipo, función y línea)
    print(bad[["name", "qc_errors"]].head(5).to_string(index=False))

C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed2\features_train.csv  ->  2000 filas x 204 columnas
C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed2\features_val.csv  ->  1000 filas x 204 columnas
shape   76 columnas | NaN medio: 0.00%
color   83 columnas | NaN medio: 0.00%
tex     38 columnas | NaN medio: 0.00%
Imágenes con algún error: 0
